# Analyse du Positionnement des Banques au Sénégal

**Notre objectif :** comprendre comment les banques sénégalaises se positionnent sur le marché à travers leurs indicateurs financiers.

**La sources :**
- `BASE_SENEGAL2.xlsx` : 24 banques, 2015-2020
- Rapports PDF BCEAO : données officielles 2020-2022, extraites par scraping + OCR

**Le plan :**
1. Collecte des données (Excel + PDF BCEAO)
2. Nettoyage et validation
3. Stockage MongoDB
4. Analyse des KPI
5. Ratios financiers
6. Score de positionnement

## 1. Installation et imports

In [ ]:
!pip install pdfplumber pytesseract pdf2image pymongo pandas openpyxl plotly -q

In [ ]:
import os
import re
import pandas as pd
import numpy as np
import pdfplumber
import pytesseract
from pdf2image import convert_from_path
from pymongo import MongoClient
import plotly.express as px
import plotly.graph_objects as go
import warnings
warnings.filterwarnings('ignore')

print('Imports OK')

## 2. Collecte : BASE_SENEGAL2.xlsx



In [ ]:
df_excel = pd.read_excel('data/BASE_SENEGAL2.xlsx')

print(f'Lignes : {len(df_excel)}')
print(f'Banques : {df_excel["Sigle"].nunique()}')
print(f'Années : {sorted(df_excel["ANNEE"].unique())}')
df_excel.head()

## 3: Rapport PDF BCEAO (Web Scraping + OCR)



In [ ]:
import requests
from bs4 import BeautifulSoup

URL = "https://www.bceao.int/fr/publications/bilans-et-comptes-de-resultat-des-banques-etablissements-financiers-et-compagnies"

try:
    reponse = requests.get(URL, headers={'User-Agent': 'Mozilla/5.0'}, timeout=15)
    soup = BeautifulSoup(reponse.text, 'html.parser')

    liens = []
    for tag in soup.find_all('a', href=True):
        href = tag['href']
        if '.pdf' in href.lower():
            if not href.startswith('http'):
                href = 'https://www.bceao.int' + href
            liens.append(href)

    print(f'{len(liens)} PDF trouvé sur le site BCEAO')
    for l in liens[:5]:
        print(' -', l)

#au cas ou les donnees ne seront plus accessible...

except Exception as e:
    print(f'Site non accessible depuis cet environnement : {e}')
    print('On utilise le PDF téléchargé manuellement dans data/')

**Étape 2 — Extraction :** 

In [ ]:
def detecter_type_page(texte):
    """Détecte si une page contient du texte ou si c'est une image scannée."""
    return 'texte' if texte and len(texte.strip()) > 50 else 'image'


def extraire_nombres(ligne):
    """Extrait les valeurs numériques d'une ligne."""
    tokens = re.findall(r'-?\d{1,3}(?:\s\d{3})*|\d+', ligne)
    valeurs = []
    for t in tokens:
        try:
            v = float(t.replace(' ', ''))
            if abs(v) < 1_000_000_000:
                valeurs.append(v)
        except ValueError:
            pass
    return valeurs


def extraire_page_texte(lignes, sigle, type_page):
    """Extrait les indicateurs depuis les lignes de texte d'une page."""
    records = []
    annees = [2020, 2021, 2022]

    mapping = {
        'bilan': ['créances sur la clientèle', 'effets publics', 'caisse, banque centrale'],
        'resultat': ['produit net bancaire', 'résultat net', 'charges générales'],
    }.get(type_page, [])

    indicateurs = {
        'créances sur la clientèle':   'creances_clientele',
        'effets publics':              'effets_publics',
        'caisse, banque centrale':     'caisse_bceao',
        'produit net bancaire':        'pnb',
        'résultat net':                'resultat_net',
        'charges générales':           'charges_exploit',
        'intérêts et produits':        'interets_produits',
        'commissions (produits)':      'commissions_prod',
        'résultat brut':               'rbe',
        'coût du risque':              'cout_risque',
    }

    for ligne in lignes:
        ll = ligne.lower()
        vals = extraire_nombres(ligne)
        if not vals:
            continue
        for mot_cle, indicateur in indicateurs.items():
            if mot_cle in ll:
                for i, annee in enumerate(annees):
                    if i < len(vals):
                        records.append({'sigle': sigle, 'annee': annee,
                                        'indicateur': indicateur, 'valeur': vals[i]})
                break
    return records


def extraire_page_ocr(image):
    """Applique l'OCR sur une image de page PDF."""
    image_gris = image.convert('L')
    texte_ocr = pytesseract.image_to_string(image_gris, lang='fra')
    return [l.strip() for l in texte_ocr.split('\n') if l.strip()]


print('Fonctions d\'extraction définies')

In [ ]:
PDF_PATH = 'data/rapport_bceao_2022.pdf'
PAGES_SENEGAL = range(266, 330)
records_bceao = []

if not os.path.exists(PDF_PATH):
    print(f'PDF non trouvé : {PDF_PATH}')
    print('Placer le rapport BCEAO dans data/ avec ce nom.')
else:
    with pdfplumber.open(PDF_PATH) as pdf:
        sigle = nom = type_page = None

        for idx in PAGES_SENEGAL:
            if idx >= len(pdf.pages):
                break

            page = pdf.pages[idx]
            texte = page.extract_text()

            # Choix de la méthode selon le type de page
            if detecter_type_page(texte) == 'texte':
                lignes = [l.strip() for l in texte.split('\n') if l.strip()]
                methode = 'texte'
            else:
                # Page scannée → OCR
                try:
                    images = convert_from_path(PDF_PATH, first_page=idx+1,
                                               last_page=idx+1, dpi=200)
                    lignes = extraire_page_ocr(images[0])
                    methode = 'ocr'
                except Exception:
                    continue

            # Détection de l'en-tête
            if len(lignes) >= 2 and 'SENEGAL' in lignes[0].upper():
                sigle = lignes[1].strip()
                type_page = None
                for l in lignes:
                    if re.search(r'Bilans?\s+2020', l, re.I):
                        type_page = 'bilan'; break
                    if re.search(r'R.sultat\s+2020', l, re.I):
                        type_page = 'resultat'; break

            if sigle and type_page:
                records = extraire_page_texte(lignes, sigle, type_page)
                records_bceao.extend(records)

    print(f'{len(records_bceao)} enregistrements extraits du PDF')

In [ ]:
if records_bceao:
    df_bceao_raw = pd.DataFrame(records_bceao)
    df_bceao_raw = df_bceao_raw.drop_duplicates(subset=['sigle','annee','indicateur'], keep='first')

    df_bceao = df_bceao_raw.pivot_table(
        index=['sigle','annee'], columns='indicateur',
        values='valeur', aggfunc='first'
    ).reset_index()
    df_bceao.columns.name = None

    # Harmonisation des sigles
    remap = {'B.H.S.':'BHS','B.I.S.':'BIS','B.N.D.E':'BNDE','B.R.M.':'BRM',
             'C.D.S.':'CDS','U.B.A.':'UBA','BOA-S':'BOA','SGSN':'SGBS',
             'NSIA BANQUE':'NSIA','CBI-SENEGAL':'CBI','BGFI BANK':'BGFI'}
    df_bceao['sigle'] = df_bceao['sigle'].replace(remap)
    df_bceao['source'] = 'BCEAO_2022'

    print(f'Données BCEAO : {df_bceao.shape[0]} lignes | {df_bceao["sigle"].nunique()} banques')
    df_bceao.head()
else:
    df_bceao = pd.DataFrame()
    print('Aucune donnée BCEAO — on utilise uniquement la base Excel')

## 4. Nettoyage et validation

Avant la fusion, on nettoie chaque source séparément, puis on valide le résultat final.

In [ ]:
# Renommage BASE_SENEGAL2
df = df_excel.rename(columns={
    'Sigle': 'sigle', 'Goupe_Bancaire': 'groupe', 'ANNEE': 'annee',
    'EMPLOI': 'emploi', 'BILAN': 'bilan', 'RESSOURCES': 'ressources',
    'FONDS.PROPRE': 'fonds_propres', 'PRODUIT.NET.BANCAIRE': 'pnb',
    'RESULTAT.NET': 'resultat_net',
    "CHARGES.GENERALES.D'EXPLOITATION": 'charges_exploit',
    "RESULTAT.BRUT.D'EXPLOITATION": 'rbe',
    'COÛT.DU.RISQUE': 'cout_risque',
    'EFFECTIF': 'effectif', 'AGENCE': 'agences',
})
df['source'] = 'BASE_SENEGAL2'

# Conversion numérique
cols_num = ['bilan','emploi','ressources','fonds_propres','pnb',
            'resultat_net','charges_exploit','rbe','cout_risque']
for col in cols_num:
    df[col] = pd.to_numeric(df[col], errors='coerce')

df['annee'] = df['annee'].astype(int)

print(f'BASE_SENEGAL2 : {len(df)} lignes')

In [ ]:
# Vérification des valeurs manquantes
taux_na = df[cols_num].isnull().mean().mul(100).round(1)
print('Taux de valeurs manquantes par indicateur (%) :')
print(taux_na.sort_values(ascending=False))

In [ ]:
# Vérification des incohérences
bilans_negatifs = df[df['bilan'] < 0]
print(f'Bilans négatifs : {len(bilans_negatifs)}')

doublons = df.duplicated(subset=['sigle','annee','source']).sum()
print(f'Doublons : {doublons}')

print(f'\nBanques : {sorted(df["sigle"].unique())}')

In [ ]:
# Fusion avec les données BCEAO si disponibles
if not df_bceao.empty:
    df_bceao_new = df_bceao[df_bceao['annee'].isin([2021, 2022])].copy()
    df = pd.concat([df, df_bceao_new], ignore_index=True)
    print(f'Après fusion : {len(df)} lignes | {sorted(df["annee"].unique())}')
else:
    print(f'Dataset final : {len(df)} lignes | {sorted(df["annee"].unique())}')

In [ ]:
# Calcul des ratios financiers
df['roa']          = (df['resultat_net'] / df['bilan'] * 100).round(2)
df['coeff_exploit_ratio'] = (df['charges_exploit'] / df['pnb'] * 100).round(2)
df['taux_transfo'] = (df['emploi'] / df['ressources'] * 100).round(2)
df['solvabilite']  = (df['fonds_propres'] / df['bilan'] * 100).round(2)

# Nettoyer les ratios aberrants
df.loc[df['coeff_exploit_ratio'].abs() > 500, 'coeff_exploit_ratio'] = np.nan

print('Ratios calculés : roa, coeff_exploit_ratio, taux_transfo, solvabilite')
df[['sigle','annee','bilan','pnb','resultat_net','roa','solvabilite']].head(10)

## 5. Stockage MongoDB

On centralise les données nettoyées dans MongoDB. Chaque ligne du DataFrame devient un document JSON.

In [ ]:
MONGO_URI = 'mongodb://localhost:27017/'  # Remplacer par ton URI Atlas si besoin

try:
    client = MongoClient(MONGO_URI, serverSelectionTimeoutMS=3000)
    client.server_info()

    db = client['banques_senegal']
    collection = db['indicateurs']

    # Nettoyage avant insertion
    collection.drop()

    # Insertion
    docs = df.where(pd.notnull(df), None).to_dict(orient='records')
    collection.insert_many(docs)

    # Index pour accélérer les requêtes
    collection.create_index([('sigle', 1), ('annee', 1)])

    print(f'{len(docs)} documents insérés dans MongoDB')
    print('Exemple :', collection.find_one({'sigle': 'CBAO'}, {'_id': 0}))

except Exception as e:
    print(f'MongoDB non disponible : {e}')
    print('Données disponibles dans le DataFrame df')

## 6. Analyse des KPI

On regarde les 5 indicateurs clés du sujet : bilan, emploi, ressources, fonds propres, et les ratios.

In [ ]:
# Classement par bilan moyen
kpi = df.groupby(['sigle','groupe']).agg(
    bilan_moy       = ('bilan',         'mean'),
    emploi_moy      = ('emploi',        'mean'),
    ressources_moy  = ('ressources',    'mean'),
    fonds_propres_moy = ('fonds_propres','mean'),
    pnb_moy         = ('pnb',           'mean'),
    resultat_moy    = ('resultat_net',  'mean'),
).round(0).reset_index().sort_values('bilan_moy', ascending=False)

print('Classement des banques par bilan moyen :')
kpi.head(10)

In [ ]:
fig = px.bar(
    kpi.head(10), x='bilan_moy', y='sigle', orientation='h',
    color='groupe', color_discrete_sequence=px.colors.qualitative.Set2,
    title='Top 10 banques par bilan moyen (MFCFA)',
    labels={'bilan_moy': 'Bilan moyen', 'sigle': ''}
)
fig.update_layout(yaxis={'categoryorder': 'total ascending'})
fig.show()

In [ ]:
# Évolution du PNB des 5 plus grandes banques
top5 = kpi.head(5)['sigle'].tolist()
df_top5 = df[df['sigle'].isin(top5)].dropna(subset=['pnb']).sort_values('annee')

fig = px.line(
    df_top5, x='annee', y='pnb', color='sigle', markers=True,
    title='Évolution du PNB — Top 5 banques',
    labels={'pnb': 'PNB (MFCFA)', 'annee': 'Année'}
)
fig.show()

## 7. Ratios financiers

Les ratios permettent de comparer des banques de tailles différentes sur une même base.

| Ratio | Formule | Lecture |
|---|---|---|
| ROA | Résultat / Bilan × 100 | Rentabilité des actifs |
| Solvabilité | Fonds propres / Bilan × 100 | Solidité financière |
| Coeff. d'exploitation | Charges / PNB × 100 | Efficacité (< 60% = bien) |
| Taux de transformation | Emploi / Ressources × 100 | Utilisation des dépôts |

In [ ]:
# ROA moyen par banque
roa_banque = df.groupby('sigle')['roa'].mean().dropna().sort_values(ascending=False).reset_index()

fig = px.bar(
    roa_banque, x='sigle', y='roa',
    color='roa', color_continuous_scale='RdYlGn',
    title='ROA moyen par banque (%)',
    labels={'roa': 'ROA (%)', 'sigle': 'Banque'}
)
fig.add_hline(y=0, line_dash='dash', line_color='red')
fig.update_layout(coloraxis_showscale=False)
fig.show()

In [ ]:
# Scatter : efficacité vs rentabilité
d_scatter = df[df['annee'] == df['annee'].max()].dropna(subset=['roa', 'coeff_exploit_ratio'])
d_scatter = d_scatter[(d_scatter['coeff_exploit_ratio'] > 0) & (d_scatter['coeff_exploit_ratio'] < 300)]

fig = px.scatter(
    d_scatter, x='coeff_exploit_ratio', y='roa',
    text='sigle', color='groupe', size='bilan',
    color_discrete_sequence=px.colors.qualitative.Set2,
    title="Efficacité vs Rentabilité",
    labels={'coeff_exploit_ratio': "Coeff. d'exploitation (%)", 'roa': 'ROA (%)'}
)
fig.add_vline(x=60, line_dash='dash', line_color='orange', annotation_text='Seuil 60%')
fig.add_hline(y=0, line_dash='dash', line_color='red')
fig.update_traces(textposition='top center')
fig.show()

## 8. Score de positionnement

On calcule un score global pour classer les banques. Il repose sur 4 critères normalisés (0 à 100) : bilan, PNB, ROA, et solvabilité.

Plus le score est élevé, mieux la banque est positionnée sur le marché.

In [ ]:
criteres = ['bilan', 'pnb', 'roa', 'solvabilite']
criteres_dispo = [c for c in criteres if c in df.columns]

synthese = df.groupby('sigle')[criteres_dispo].mean().dropna()

# Normalisation 0-100
for col in criteres_dispo:
    mn, mx = synthese[col].min(), synthese[col].max()
    synthese[f'{col}_score'] = ((synthese[col] - mn) / (mx - mn) * 100).round(1) if mx > mn else 50.0

score_cols = [f'{c}_score' for c in criteres_dispo]
synthese['score_global'] = synthese[score_cols].mean(axis=1).round(1)
synthese['classement']   = synthese['score_global'].rank(ascending=False).astype(int)

classement_final = synthese[['score_global', 'classement']].sort_values('classement')
print('Classement final des banques :')
classement_final

In [ ]:
# Visualisation du score global
df_score = classement_final.reset_index().sort_values('score_global', ascending=True)

fig = px.bar(
    df_score, x='score_global', y='sigle', orientation='h',
    color='score_global', color_continuous_scale='Blues',
    title='Score de positionnement global (0-100)',
    labels={'score_global': 'Score', 'sigle': ''}
)
fig.update_layout(coloraxis_showscale=False)
fig.show()